# KantBench GRPO Training

Train a language model to play game-theory games using **Group Relative Policy Optimization (GRPO)** via HuggingFace TRL.

The model learns optimal strategies by playing full episodes against the [KantBench](https://openenv-community-kantbench.hf.space) environment — an OpenEnv server hosting **104 game-theory games**, **17 opponent strategies**, and **9 composable meta-game variants** (cheap talk, binding commitments, constitutional governance, etc.).

**Reward signal**: composite score of payoff + cooperation rate + Pareto efficiency + fairness.

**Variant composition**: 30% of training samples dynamically compose a variant (e.g. Prisoner's Dilemma + cheap_talk) to train on meta-gaming scenarios.

In [ ]:
!pip install -q trl transformers datasets accelerate websockets pydantic

In [ ]:
import json
import random
import logging
from dataclasses import dataclass
from typing import Any

import torch
import websockets.sync.client as ws_client
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoTokenizer

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Configuration

In [ ]:
KANTBENCH_URL = "https://openenv-community-kantbench.hf.space"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # fits on Colab T4/A100

SYSTEM_PROMPT = (
    "You are playing a game-theory game. Analyse the situation and choose "
    "the best action. Respond with ONLY the action name, nothing else."
)

# Full KantBench game library (104 games across 20+ game-theoretic domains)
GAMES = [
    # Classic matrix games
    "prisoners_dilemma", "stag_hunt", "hawk_dove", "battle_of_the_sexes",
    "matching_pennies", "rock_paper_scissors", "pure_coordination", "deadlock", "harmony",
    # Sequential games
    "dictator", "centipede", "stackelberg",
    # Auctions
    "first_price_auction", "vickrey_auction", "allpay_auction",
    # N-player games
    "tragedy_of_commons", "volunteer_dilemma", "el_farol",
    # Generated / random matrices
    "random_symmetric_3x3", "random_asymmetric_3x3", "random_zero_sum_3x3", "random_coordination_3x3",
    # Information games — signaling
    "beer_quiche", "spence_signaling", "cheap_talk", "lemon_market", "bayesian_persuasion",
    # Information games — contracts
    "moral_hazard", "screening", "gift_exchange",
    # Information games — communication
    "cheap_talk_pd", "binding_commitment", "correlated_equilibrium", "focal_point", "mediated_game",
    # Information games — Bayesian
    "global_game", "jury_voting", "information_cascade", "adverse_selection_insurance",
    # Information games — network
    "security_game", "link_formation", "trust_with_punishment", "dueling_game",
    # Market / competition
    "cournot", "bertrand", "hotelling", "double_auction", "dollar_auction",
    "tullock_contest", "colonel_blotto", "war_of_attrition", "beauty_contest",
    "nash_demand", "rubinstein_bargaining", "minority_game",
    # Voting / governance
    "approval_voting", "median_voter", "weighted_voting",
    # PD variants
    "optional_pd", "continuous_pd", "discounted_pd", "evolutionary_pd",
    "finitely_repeated_pd", "stochastic_pd", "asymmetric_pd", "donation_game",
    # Cooperative games
    "core_divide_dollar", "shapley_allocation", "stable_matching", "divide_and_choose",
    # Dynamic / advanced
    "markov_game", "bank_run", "global_stag_hunt", "entry_deterrence",
    "preemption_game", "inspection_game", "peace_war", "risk_dominance",
    "parameterized_chicken", "penalty_shootout", "rpsls",
    "friend_or_foe", "unscrupulous_diner", "war_of_gifts",
    "hawk_dove_bourgeois", "travelers_dilemma",
    # Computed payoff games
    "ultimatum", "trust", "public_goods", "threshold_public_goods",
    # Meta-games (pre-composed)
    "rule_proposal_prisoners_dilemma", "rule_proposal_stag_hunt", "rule_proposal_hawk_dove",
    "rule_signal_prisoners_dilemma", "rule_signal_stag_hunt", "rule_signal_hawk_dove",
    "gossip_prisoners_dilemma", "gossip_stag_hunt", "gossip_hawk_dove",
    # Adaptive factory games
    "adaptive_prisoners_dilemma", "arms_race", "market_dynamics",
    "reputation_payoffs", "trust_erosion",
]

# All 17 opponent strategies
STRATEGIES = [
    # General matrix-game strategies
    "random", "always_cooperate", "always_defect",
    "tit_for_tat", "tit_for_two_tats", "grudger", "pavlov",
    "suspicious_tit_for_tat", "generous_tit_for_tat",
    "adaptive", "mixed",
    # Game-specific strategies
    "ultimatum_fair", "ultimatum_low",
    "trust_fair", "trust_generous",
    "public_goods_fair", "public_goods_free_rider",
]

# Dynamic variant composition — applied server-side on top of base games
TRAINABLE_VARIANTS = [
    "cheap_talk",            # Non-binding messaging phase
    "exit",                  # Safe exit option with fixed payoff
    "binding_commitment",    # Costly commitment mechanism
    "constitutional",        # Persistent governance rules
    "noisy_actions",         # Action trembles
    "noisy_payoffs",         # Payoff noise
    "rule_proposal",         # Binding per-round rule proposals
    "rule_signal",           # Non-binding rule signals
    "gossip",                # Reputation signaling
]

# Base games suitable for variant composition (2-player matrix games)
VARIANT_BASE_GAMES = ["prisoners_dilemma", "stag_hunt", "hawk_dove"]

# Fraction of dataset samples using dynamic variant composition
VARIANT_FRACTION = 0.3

print(f"Games: {len(GAMES)} | Strategies: {len(STRATEGIES)} | Variants: {len(TRAINABLE_VARIANTS)}")

## KantBench Client

Minimal OpenEnv WebSocket client for the KantBench environment.

In [ ]:
@dataclass
class Observation:
    game_name: str
    game_description: str
    available_moves: list[str]
    cumulative_score: float
    round_number: int
    max_rounds: int
    history: list[dict]
    done: bool = False


class KantBenchClient:
    """Lightweight WebSocket client for the KantBench OpenEnv server."""

    def __init__(self, base_url: str = KANTBENCH_URL):
        ws_url = base_url.replace("https://", "wss://").replace("http://", "ws://")
        self.ws_url = f"{ws_url}/ws"
        self.ws = None

    def connect(self):
        self.ws = ws_client.connect(self.ws_url)

    def close(self):
        if self.ws:
            self.ws.close()

    def _send(self, msg: dict) -> dict:
        self.ws.send(json.dumps(msg))
        return json.loads(self.ws.recv())

    def _parse_obs(self, payload: dict) -> Observation:
        obs = payload.get("observation", {})
        return Observation(
            game_name=obs.get("game_name", ""),
            game_description=obs.get("game_description", ""),
            available_moves=obs.get("available_moves", []),
            cumulative_score=obs.get("cumulative_score", 0.0),
            round_number=obs.get("round_number", 0),
            max_rounds=obs.get("max_rounds", 10),
            history=obs.get("history", []),
            done=payload.get("done", False),
        )

    def reset(self, game: str, strategy: str, variant: str | None = None) -> Observation:
        msg = {"type": "reset", "game": game, "strategy": strategy}
        if variant:
            msg["variant"] = variant
        resp = self._send(msg)
        return self._parse_obs(resp)

    def step(self, move: str) -> Observation:
        resp = self._send({"type": "step", "move": move})
        return self._parse_obs(resp)

## Prompt Builder & Action Parser

In [ ]:
def build_prompt(obs: Observation) -> str:
    """Build a structured prompt from a game observation."""
    sections = []
    sections.append(f"[Game]\n{obs.game_name}\n{obs.game_description}")

    if obs.history:
        lines = []
        for h in obs.history[-5:]:
            lines.append(
                f"Round {h.get('round', '?')}"
                f" | You: {h.get('your_move', '?')}"
                f" | Opp: {h.get('opponent_move', '?')}"
                f" | Payoff: {h.get('your_payoff', '?')}"
            )
        sections.append("[History]\n" + "\n".join(lines))

    sections.append(f"[Scores]\nYour score: {obs.cumulative_score}\nRound: {obs.round_number}/{obs.max_rounds}")
    sections.append("[Actions]\n" + "\n".join(f"- {a}" for a in obs.available_moves))
    sections.append(f"[Instruction]\n{SYSTEM_PROMPT}")
    return "\n\n".join(sections)


def parse_action(response: str, available: list[str]) -> str:
    """Parse action from LLM output: exact -> case-insensitive -> substring -> random."""
    s = response.strip()
    if s in available:
        return s
    for a in available:
        if a.lower() == s.lower():
            return a
    for a in available:
        if a.lower() in s.lower():
            return a
    return random.choice(available)

## Dataset Generation

Generate training prompts by playing random partial games on the KantBench server. 30% of samples use **dynamic variant composition** — a meta-game variant (e.g. cheap_talk, constitutional) is composed on top of a base game to create richer strategic scenarios.

In [ ]:
def build_dataset(n_samples: int = 512) -> Dataset:
    client = KantBenchClient()
    client.connect()
    samples = []

    while len(samples) < n_samples:
        # Decide whether to use dynamic variant composition
        use_variant = random.random() < VARIANT_FRACTION
        if use_variant:
            game = random.choice(VARIANT_BASE_GAMES)
            variant = random.choice(TRAINABLE_VARIANTS)
        else:
            game = random.choice(GAMES)
            variant = None

        strategy = random.choice(STRATEGIES)
        try:
            obs = client.reset(game=game, strategy=strategy, variant=variant)

            # Play 0..N-1 random rounds to get diverse game states
            for _ in range(random.randint(0, max(obs.max_rounds - 1, 0))):
                obs = client.step(random.choice(obs.available_moves))
                if obs.done:
                    obs = client.reset(game=game, strategy=strategy, variant=variant)
                    break

            samples.append({
                "prompt": build_prompt(obs),
                "game_key": game,
                "strategy": strategy,
                "variant": variant or "",
                "available_moves": obs.available_moves,
            })
        except Exception as e:
            logger.debug(f"Skip {game}/{strategy} (variant={variant}): {e}")
            continue

    client.close()
    return Dataset.from_list(samples)


dataset = build_dataset(512)
variant_count = sum(1 for v in dataset["variant"] if v)
print(f"Generated {len(dataset)} training prompts")
print(f"  Variant samples: {variant_count} ({variant_count * 100 // len(dataset)}%)")
print(f"Example prompt:\n{dataset[0]['prompt'][:300]}...")

## Reward Function

For each LLM completion, play a **full episode** using the predicted move as a consistent strategy. The reward is a composite of:
- **Payoff**: normalised game score
- **Cooperation**: rate of cooperative actions
- **Pareto efficiency**: joint welfare proxy
- **Fairness**: score parity between players

In [ ]:
def episode_reward(player_score, opponent_score, cooperation_rate, total_rounds):
    """Composite reward: equal-weighted cooperation + pareto + fairness + neutral baselines."""
    w = 0.2  # 5 components, equal weight

    coop = cooperation_rate
    pareto = min(1.0, max(0.0, (player_score + opponent_score) / total_rounds)) if total_rounds > 0 else 0.0
    denom = abs(player_score) + abs(opponent_score)
    fairness = 1.0 - abs(player_score - opponent_score) / denom if denom > 0 else 1.0

    return w * coop + w * pareto + w * fairness + w * 0.5 + w * 0.5  # exploit_resist + adaptability = neutral


# Persistent client for reward computation
reward_client = KantBenchClient()
reward_client.connect()


def kantbench_reward_fn(completions, prompts, **kwargs):
    """GRPO reward: play full episodes via KantBench and score them."""
    rewards = []
    game_keys = kwargs.get("game_key", ["prisoners_dilemma"] * len(completions))
    strategies = kwargs.get("strategy", ["tit_for_tat"] * len(completions))
    variants = kwargs.get("variant", [""] * len(completions))
    moves_batch = kwargs.get("available_moves", [["cooperate", "defect"]] * len(completions))

    for completion, game, strat, variant, moves in zip(
        completions, game_keys, strategies, variants, moves_batch
    ):
        action = parse_action(completion.strip(), moves)
        try:
            obs = reward_client.reset(game=game, strategy=strat, variant=variant or None)
            while not obs.done:
                obs = reward_client.step(action)

            coop_actions = {"cooperate", "stag", "dove", "contribute"}
            coop_rate = sum(1 for h in obs.history if any(c in h.get("your_move", "") for c in coop_actions)) / max(len(obs.history), 1)
            opp_score = sum(h.get("opponent_payoff", 0.0) for h in obs.history)

            rewards.append(episode_reward(obs.cumulative_score, opp_score, coop_rate, obs.round_number))
        except Exception:
            rewards.append(-1.0)

    return rewards


def format_reward_fn(completions, prompts, **kwargs):
    """Bonus reward for clean, exact-match action output."""
    rewards = []
    moves_batch = kwargs.get("available_moves", [["cooperate", "defect"]] * len(completions))
    for completion, moves in zip(completions, moves_batch):
        s = completion.strip()
        if s in moves:
            rewards.append(1.0)
        elif s.lower() in [m.lower() for m in moves]:
            rewards.append(0.5)
        elif any(m.lower() in s.lower() for m in moves):
            rewards.append(0.1)
        else:
            rewards.append(-0.5)
    return rewards

## Training

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Format prompts with chat template
def format_prompt(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["prompt"]},
    ]
    return {"prompt": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)}

dataset = dataset.map(format_prompt)
print(f"Dataset ready: {len(dataset)} samples")

In [ ]:
config = GRPOConfig(
    output_dir="./kantbench-grpo",
    num_generations=4,
    max_completion_length=32,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=3e-6,
    lr_scheduler_type="constant_with_warmup",
    warmup_steps=50,
    max_steps=1000,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    report_to="none",
    generation_kwargs={"temperature": 0.7},
)

# Stop generation at newline to enforce single-action output
nl_token = tokenizer.encode("\n", add_special_tokens=False)
if nl_token:
    config.generation_kwargs["eos_token_id"] = [tokenizer.eos_token_id, nl_token[0]]

trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=[kantbench_reward_fn, format_reward_fn],
    args=config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("Starting GRPO training...")
trainer.train()

In [ ]:
trainer.save_model("./kantbench-grpo")
print("Model saved to ./kantbench-grpo")

## Push to Hub (optional)

In [ ]:
# trainer.push_to_hub("your-username/kantbench-qwen2.5-3b")